## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, 

### To Do List:

1) download data from bezrealitky.cz and reality.idnes.cz, and from sreality.cz on other cities
2) create heatmap based on longitude and latitude - DONE
3) add property popups to map?
4) download info on airbnb prices
5) conduct a simple analysis of rental price determinants

In [ ]:
# import packages

import json
import pandas as pd
import os
import requests 
import pandas as pd 
import time
import re 
import random 
import folium
from folium.plugins import HeatMap, MarkerCluster
import math

In [ ]:
# get df from last request
df = pd.read_json("df.json")
df = pd.DataFrame(df)


In [ ]:
# or get newest df, takes about 7 minutes
from function_scripts import request_sreality_all
df = request_sreality_all() 
df.to_json("df.json", orient="records")

In [ ]:
df.shape
df.head()

In [ ]:
def name_to_area(nm):
    splitted_str = nm.split()
    m2_idx = splitted_str.index('m²')
    return int(splitted_str[m2_idx - 1])

df['area'] = df.name.apply(name_to_area)

df['flat_type'] = df.name.apply(lambda x: x.split()[2])
print(df)

In [ ]:
from function_scripts import get_link_and_image
df = get_link_and_image(df)
display(df)

In [ ]:
columns_to_keep = ['locality', 'price', 'name', 'flat_type','area','gps','hash_id','exclusively_at_rk','url','image']
df_clean = df[columns_to_keep].copy()



df_clean.head()

In [ ]:
df_clean[['lat', 'lon']] = df_clean.gps.apply(lambda x: pd.Series({'lat': x['lat'], 'lon': x['lon']}))
df_clean.head()

In [ ]:
df_heatmap = df_clean[['lat', 'lon', 'price']].copy()
df_property = df_clean[['lat', 'lon', 'price','locality', 'flat_type', 'area', 'url', 'image']].copy()

In [ ]:
# base map, centered on CZ
m = folium.Map(location=(49.75, 15.40), zoom_start = 8)

# heatmap layer
heat_layer = folium.FeatureGroup(name="Heat Map", show=True)

HeatMap(
    df_heatmap,
    min_opacity=0.4,
    blur=18
).add_to(heat_layer)

heat_layer.add_to(m)

# property popups layer
property_layer = folium.FeatureGroup(
    name="Properties",
    show=False
)

marker_cluster = MarkerCluster().add_to(property_layer)

# create popups with variables from df_property
for _, row in df_property.iterrows():

    popup_html = f"""
    <b>{row['price']:,} CZK</b><br>
    {row['locality']}<br>
    {row['flat_type']}<br>
    {row['area']} m²<br><br>
    <img src="{row['image']}" width="200"><br>
    <a href="{row['url']}" target="_blank">Open listing</a>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(marker_cluster)

property_layer.add_to(m)

# layer control
folium.LayerControl(collapsed=False).add_to(m)

# save
m.save("heatmap.html")